In [7]:
# ── Cell 1: Import Libraries and Load Cleaned Data ────────────────────

import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import joblib
import os

# Auto-detect environment (Colab or Local)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    COLAB = True
except ImportError:
    COLAB = False

if COLAB and os.path.exists('/content/drive/MyDrive/Machine_Learning'):
    DATA_PATH   = '/content/drive/MyDrive/Machine_Learning/data/'
    MODELS_PATH = '/content/drive/MyDrive/Machine_Learning/models/'
    print("Environment: Google Colab + Google Drive")
else:
    DATA_PATH   = '../data/'
    MODELS_PATH = '../models/'
    print("Environment: Local")

print(f"Data path  : {DATA_PATH}")
print(f"Models path: {MODELS_PATH}")

# Load cleaned data saved from 01_EDA.ipynb
# This avoids re-running the entire EDA notebook every time
df_2078 = pd.read_csv(DATA_PATH + 'cleaned_2078.csv',
                       index_col=0, parse_dates=True).squeeze()
df_2079 = pd.read_csv(DATA_PATH + 'cleaned_2079.csv',
                       index_col=0, parse_dates=True).squeeze()

print(f"\nSite 2078 - Total timestamps: {len(df_2078)}")
print(f"Site 2079 - Total timestamps: {len(df_2079)}")
print(f"Date range: {df_2078.index.min()} → {df_2078.index.max()}")
print(f"Any missing values - 2078: {df_2078.isnull().sum()}, 2079: {df_2079.isnull().sum()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Environment: Google Colab + Google Drive
Data path  : /content/drive/MyDrive/Machine_Learning/data/
Models path: /content/drive/MyDrive/Machine_Learning/models/

Site 2078 - Total timestamps: 300864
Site 2079 - Total timestamps: 300864
Date range: 2013-01-01 00:00:00 → 2021-07-31 23:45:00
Any missing values - 2078: 0, 2079: 0


In [8]:
# ── Cell 2: Normalisation ─────────────────────────────────────────────

# LSTM/GRU models are sensitive to the scale of input data.
# MinMaxScaler scales all values to the range [0, 1].
# IMPORTANT: fit the scaler on training data only to prevent data leakage.
# We will fit after splitting, but here we initialise the scalers.

# Convert Series to numpy array for sklearn compatibility
data_2078 = df_2078.values.reshape(-1, 1)  # Shape: (300864, 1)
data_2079 = df_2079.values.reshape(-1, 1)  # Shape: (300864, 1)

print(f"Data shape - 2078: {data_2078.shape}")
print(f"Data shape - 2079: {data_2079.shape}")
print(f"\nBefore normalisation:")
print(f"  2078 - Min: {data_2078.min():.1f}, Max: {data_2078.max():.1f}, Mean: {data_2078.mean():.1f}")
print(f"  2079 - Min: {data_2079.min():.1f}, Max: {data_2079.max():.1f}, Mean: {data_2079.mean():.1f}")

Data shape - 2078: (300864, 1)
Data shape - 2079: (300864, 1)

Before normalisation:
  2078 - Min: 0.0, Max: 1577.0, Mean: 692.2
  2079 - Min: 0.0, Max: 1765.0, Mean: 658.1


In [9]:
# ── Cell 3: Train / Validation / Test Split ───────────────────────────

# Time series data must be split in chronological order (no random shuffling!)
# Splitting randomly would cause data leakage (future data leaking into training).
#
# Split ratio: 70% Train / 15% Validation / 15% Test
# Train:      2013-01-01 → 2019-01-03  (learning normal traffic patterns)
# Validation: 2019-01-03 → 2020-04-17  (hyperparameter tuning)
# Test:       2020-04-17 → 2021-07-31  (final evaluation)

total      = len(data_2078)
train_size = int(total * 0.70)  # 70% for training
val_size   = int(total * 0.15)  # 15% for validation
test_size  = total - train_size - val_size  # remaining 15% for test

print(f"Total timestamps : {total}")
print(f"Train size       : {train_size} ({train_size/total*100:.1f}%)")
print(f"Validation size  : {val_size}   ({val_size/total*100:.1f}%)")
print(f"Test size        : {test_size}  ({test_size/total*100:.1f}%)")

# Split data for both sites
train_2078 = data_2078[:train_size]
val_2078   = data_2078[train_size:train_size + val_size]
test_2078  = data_2078[train_size + val_size:]

train_2079 = data_2079[:train_size]
val_2079   = data_2079[train_size:train_size + val_size]
test_2079  = data_2079[train_size + val_size:]

# Print corresponding date ranges for each split
print(f"\nDate ranges:")
print(f"  Train : {df_2078.index[0]}  → {df_2078.index[train_size-1]}")
print(f"  Val   : {df_2078.index[train_size]} → {df_2078.index[train_size+val_size-1]}")
print(f"  Test  : {df_2078.index[train_size+val_size]} → {df_2078.index[-1]}")

Total timestamps : 300864
Train size       : 210604 (70.0%)
Validation size  : 45129   (15.0%)
Test size        : 45131  (15.0%)

Date ranges:
  Train : 2013-01-01 00:00:00  → 2019-01-03 18:45:00
  Val   : 2019-01-03 19:00:00 → 2020-04-17 21:00:00
  Test  : 2020-04-17 21:15:00 → 2021-07-31 23:45:00


In [10]:
# ── Cell 4: Fit Scaler on Training Data Only ──────────────────────────

# IMPORTANT: Scaler must be fit ONLY on training data.
# Fitting on the full dataset would cause data leakage,
# as the model would indirectly "know" the min/max of future data.

scaler_2078 = MinMaxScaler(feature_range=(0, 1))
scaler_2079 = MinMaxScaler(feature_range=(0, 1))

# Fit on training data only, then transform all splits
train_2078_scaled = scaler_2078.fit_transform(train_2078)
val_2078_scaled   = scaler_2078.transform(val_2078)
test_2078_scaled  = scaler_2078.transform(test_2078)

train_2079_scaled = scaler_2079.fit_transform(train_2079)
val_2079_scaled   = scaler_2079.transform(val_2079)
test_2079_scaled  = scaler_2079.transform(test_2079)

# Save scalers to disk for later use during model evaluation
# We need the scaler to inverse_transform predictions back to original scale
os.makedirs(MODELS_PATH, exist_ok=True)
joblib.dump(scaler_2078, MODELS_PATH + 'scaler_2078.pkl')
joblib.dump(scaler_2079, MODELS_PATH + 'scaler_2079.pkl')

print("Scalers fitted and saved!")
print(f"\nAfter normalisation (training data):")
print(f"  2078 - Min: {train_2078_scaled.min():.3f}, Max: {train_2078_scaled.max():.3f}")
print(f"  2079 - Min: {train_2079_scaled.min():.3f}, Max: {train_2079_scaled.max():.3f}")

Scalers fitted and saved!

After normalisation (training data):
  2078 - Min: 0.000, Max: 1.000
  2079 - Min: 0.000, Max: 1.000


In [11]:
# ── Cell 5: Create Sliding Window Sequences ───────────────────────────

# LSTM/GRU models require input in the format: (samples, time_steps, features)
# We use a sliding window approach:
#   Input  (X): past 96 time steps (96 × 15min = 24 hours of history)
#   Output (y): the next single time step (predicting 15 min ahead)
#
# Example with seq_len=4 for illustration:
#   [t1, t2, t3, t4] → predict t5
#   [t2, t3, t4, t5] → predict t6
#   [t3, t4, t5, t6] → predict t7

SEQ_LEN = 96  # 24 hours of history (96 × 15min)

def create_sequences(data, seq_len):
    """
    Convert a 1D time series array into (X, y) pairs using sliding window.

    Args:
        data    : normalised numpy array of shape (n, 1)
        seq_len : number of past time steps to use as input

    Returns:
        X : shape (samples, seq_len, 1)
        y : shape (samples, 1)
    """
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i : i + seq_len])   # Past seq_len steps as input
        y.append(data[i + seq_len])        # Next step as target
    return np.array(X), np.array(y)

# Create sequences for both sites and all splits
X_train_2078, y_train_2078 = create_sequences(train_2078_scaled, SEQ_LEN)
X_val_2078,   y_val_2078   = create_sequences(val_2078_scaled,   SEQ_LEN)
X_test_2078,  y_test_2078  = create_sequences(test_2078_scaled,  SEQ_LEN)

X_train_2079, y_train_2079 = create_sequences(train_2079_scaled, SEQ_LEN)
X_val_2079,   y_val_2079   = create_sequences(val_2079_scaled,   SEQ_LEN)
X_test_2079,  y_test_2079  = create_sequences(test_2079_scaled,  SEQ_LEN)

print(f"Sequence length: {SEQ_LEN} steps = 24 hours")
print(f"\nSite 2078:")
print(f"  X_train: {X_train_2078.shape}, y_train: {y_train_2078.shape}")
print(f"  X_val  : {X_val_2078.shape},   y_val  : {y_val_2078.shape}")
print(f"  X_test : {X_test_2078.shape},  y_test : {y_test_2078.shape}")
print(f"\nSite 2079:")
print(f"  X_train: {X_train_2079.shape}, y_train: {y_train_2079.shape}")
print(f"  X_val  : {X_val_2079.shape},   y_val  : {y_val_2079.shape}")
print(f"  X_test : {X_test_2079.shape},  y_test : {y_test_2079.shape}")

Sequence length: 96 steps = 24 hours

Site 2078:
  X_train: (210508, 96, 1), y_train: (210508, 1)
  X_val  : (45033, 96, 1),   y_val  : (45033, 1)
  X_test : (45035, 96, 1),  y_test : (45035, 1)

Site 2079:
  X_train: (210508, 96, 1), y_train: (210508, 1)
  X_val  : (45033, 96, 1),   y_val  : (45033, 1)
  X_test : (45035, 96, 1),  y_test : (45035, 1)


In [13]:
# ── Cell 6: Save Preprocessed Data to Disk ───────────────────────────

# Save all sequences to disk so LSTM and GRU notebooks can load them directly
# without re-running the preprocessing pipeline every time.
# Both models (LSTM and GRU) will use exactly the same data for fair comparison.

os.makedirs(DATA_PATH, exist_ok=True)

# Save Site 2078 sequences
np.save(DATA_PATH + 'X_train_2078.npy', X_train_2078)
np.save(DATA_PATH + 'y_train_2078.npy', y_train_2078)
np.save(DATA_PATH + 'X_val_2078.npy',   X_val_2078)
np.save(DATA_PATH + 'y_val_2078.npy',   y_val_2078)
np.save(DATA_PATH + 'X_test_2078.npy',  X_test_2078)
np.save(DATA_PATH + 'y_test_2078.npy',  y_test_2078)

# Save Site 2079 sequences
np.save(DATA_PATH + 'X_train_2079.npy', X_train_2079)
np.save(DATA_PATH + 'y_train_2079.npy', y_train_2079)
np.save(DATA_PATH + 'X_val_2079.npy',   X_val_2079)
np.save(DATA_PATH + 'y_val_2079.npy',   y_val_2079)
np.save(DATA_PATH + 'X_test_2079.npy',  X_test_2079)
np.save(DATA_PATH + 'y_test_2079.npy',  y_test_2079)

print(f"All preprocessed data saved to {DATA_PATH}")
print(f"\nFiles saved:")
print(f"  Site 2078: X_train, y_train, X_val, y_val, X_test, y_test")
print(f"  Site 2079: X_train, y_train, X_val, y_val, X_test, y_test")
print(f"\nData summary:")
print(f"  Training samples  : {X_train_2078.shape[0]:,}")
print(f"  Validation samples: {X_val_2078.shape[0]:,}")
print(f"  Test samples      : {X_test_2078.shape[0]:,}")
print(f"  Sequence length   : {X_train_2078.shape[1]} steps (24 hours)")
print(f"  Features          : {X_train_2078.shape[2]}")

# Verify all files saved correctly
print("\nVerifying saved files:")
files_to_check = [
    'X_train_2078.npy', 'y_train_2078.npy',
    'X_val_2078.npy',   'y_val_2078.npy',
    'X_test_2078.npy',  'y_test_2078.npy',
    'X_train_2079.npy', 'y_train_2079.npy',
    'X_val_2079.npy',   'y_val_2079.npy',
    'X_test_2079.npy',  'y_test_2079.npy',
]

for f in files_to_check:
    path = DATA_PATH + f
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024*1024)
        print(f"  ✓ {f} ({size_mb:.1f} MB)")
    else:
        print(f"  ✗ {f} — NOT FOUND!")


# Check file modification time
import os
import datetime

for f in files_to_check:
    path = DATA_PATH + f
    if os.path.exists(path):
        mod_time = os.path.getmtime(path)
        mod_str = datetime.datetime.fromtimestamp(mod_time).strftime('%Y-%m-%d %H:%M:%S')
        size_mb = os.path.getsize(path) / (1024*1024)
        print(f"{f}: {mod_str} ({size_mb:.1f} MB)")

All preprocessed data saved to /content/drive/MyDrive/Machine_Learning/data/

Files saved:
  Site 2078: X_train, y_train, X_val, y_val, X_test, y_test
  Site 2079: X_train, y_train, X_val, y_val, X_test, y_test

Data summary:
  Training samples  : 210,508
  Validation samples: 45,033
  Test samples      : 45,035
  Sequence length   : 96 steps (24 hours)
  Features          : 1

Verifying saved files:
  ✓ X_train_2078.npy (154.2 MB)
  ✓ y_train_2078.npy (1.6 MB)
  ✓ X_val_2078.npy (33.0 MB)
  ✓ y_val_2078.npy (0.3 MB)
  ✓ X_test_2078.npy (33.0 MB)
  ✓ y_test_2078.npy (0.3 MB)
  ✓ X_train_2079.npy (154.2 MB)
  ✓ y_train_2079.npy (1.6 MB)
  ✓ X_val_2079.npy (33.0 MB)
  ✓ y_val_2079.npy (0.3 MB)
  ✓ X_test_2079.npy (33.0 MB)
  ✓ y_test_2079.npy (0.3 MB)
X_train_2078.npy: 2026-05-29 23:34:30 (154.2 MB)
y_train_2078.npy: 2026-05-29 23:34:30 (1.6 MB)
X_val_2078.npy: 2026-05-29 23:34:30 (33.0 MB)
y_val_2078.npy: 2026-05-29 23:34:30 (0.3 MB)
X_test_2078.npy: 2026-05-29 23:34:30 (33.0 MB)
y_test